In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # ARC LAB — WattTime Authentication
# MAGIC **Defines `get_watttime_token()` and confirms credentials are working.**
# MAGIC
# MAGIC Run this notebook standalone to test auth, or note that the ingest notebooks
# MAGIC include their own inline token calls (WattTime tokens expire after 30 minutes).

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Load Credentials from Secrets

# COMMAND ----------

USERNAME = dbutils.secrets.get(scope="arc-lab", key="watttime-username")
PASSWORD = dbutils.secrets.get(scope="arc-lab", key="watttime-password")

# Confirm secrets loaded without revealing values
print(f"✓ Username loaded: {USERNAME}")
print(f"✓ Password loaded: {'*' * len(PASSWORD)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Define Auth Function

# COMMAND ----------

import requests
from datetime import datetime, timezone

def get_watttime_token(username: str, password: str) -> str:
    """
    Authenticates with the WattTime API and returns a bearer token.
    Token is valid for 30 minutes — re-call this function in long-running notebooks.
    """
    url = "https://api.watttime.org/login"

    try:
        response = requests.get(url, auth=(username, password), timeout=10)
        response.raise_for_status()
        token = response.json().get("token")
        if not token:
            raise ValueError("No token in response — check credentials")
        return token

    except requests.exceptions.HTTPError:
        raise Exception(f"Auth failed — HTTP {response.status_code}. Check username/password in arc-lab secrets.")
    except requests.exceptions.ConnectionError:
        raise Exception("Auth failed — connection error. Check network access to api.watttime.org.")
    except requests.exceptions.Timeout:
        raise Exception("Auth failed — request timed out after 10 seconds.")

print("✓ get_watttime_token() defined")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Test the Connection

# COMMAND ----------

token = get_watttime_token(USERNAME, PASSWORD)

print(f"✓ Token received at {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"✓ Token preview: {token[:12]}...")
print(f"\nWattTime auth working. Ready to ingest.")

✓ Username loaded: [REDACTED]
✓ Password loaded: ***************
✓ get_watttime_token() defined
✓ Token received at 2026-03-08 05:20:26 UTC
✓ Token preview: eyJhbGciOiJS...

WattTime auth working. Ready to ingest.
